# ETL Step by Step

This notebook demonstrates a full ETL flow for this project:

1. Extract mixed-format source data from `source_data`
2. Transform the operational tables into warehouse dimensions and facts
3. Load the results into the warehouse database defined by `DATABASE_URL` in `.env`

Before running this notebook:

- make sure `.env` contains a valid `DATABASE_URL`
- make sure the database schema has been migrated with Alembic
- regenerate source data first if needed with `uv run source_data/generate_source_data.py`


In [18]:
from __future__ import annotations

import sys
import sqlite3
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "source_data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from database_schema.schema import Base

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 1. Locate the project root and load environment variables

This cell keeps the notebook flexible whether it is launched from the project root or from the `notebooks` folder.


In [19]:
import os

SOURCE_ROOT = PROJECT_ROOT / 'source_data'
MANIFEST_PATH = SOURCE_ROOT / 'json' / '_manifest.json'

load_dotenv(PROJECT_ROOT / '.env')
DATABASE_URL = os.getenv('DATABASE_URL')

if not DATABASE_URL:
    raise ValueError('DATABASE_URL is empty. Fill it in inside .env before running this notebook.')

if DATABASE_URL.startswith('postgresql+psycopg://'):
    DATABASE_URL = DATABASE_URL.replace('postgresql+psycopg://', 'postgresql+psycopg2://', 1)

print(f'Project root: {PROJECT_ROOT}')
print(f'Manifest: {MANIFEST_PATH}')
print(f"Database URL loaded: {'yes' if DATABASE_URL else 'no'}")


Project root: c:\Users\bimyu\Documents\Projects\data-warehouse-final
Manifest: c:\Users\bimyu\Documents\Projects\data-warehouse-final\source_data\json\_manifest.json
Database URL loaded: yes


## 2. Read the manifest and build source readers

The generated source data is intentionally mixed across JSON, CSV, and SQLite. The manifest tells us where each source table lives.


In [20]:
manifest = pd.read_json(MANIFEST_PATH).iloc[0].to_dict()
source_tables = manifest["tables"]

source_index = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "format": details["format"],
            "path": details["path"],
            "rows": details["rows"],
        }
        for table_name, details in source_tables.items()
    ]
).sort_values(["format", "table_name"]).reset_index(drop=True)

source_index


,table_name,format,path,rows
0,hr_employees,csv,csv\hr_employees.csv,8
1,sales_employee_segment_lookup,csv,csv\sales_employee_segment_lookup.csv,8
2,wms_inventory_transactions,csv,csv\wms_inventory_transactions.csv,2000
3,wms_warehouses,csv,csv\wms_warehouses.csv,5
4,crm_customers,json,json\crm_customers.json,12
5,crm_regions,json,json\crm_regions.json,6
6,customer_region_snapshot,json,json\customer_region_snapshot.json,12
7,lgs_shipments,json,json\lgs_shipments.json,45
8,segment_reference,json,json\segment_reference.json,4
9,so_order_items,json,json\so_order_items.json,151


In [21]:
def relative_source_path(value: str) -> Path:
    return SOURCE_ROOT / Path(value.replace("\\", "/"))


def load_source_table(table_name: str) -> pd.DataFrame:
    details = source_tables[table_name]
    table_format = details["format"]
    table_path = relative_source_path(details["path"])

    if table_format == "json":
        return pd.read_json(table_path)
    if table_format == "csv":
        return pd.read_csv(table_path)
    if table_format == "sqlite":
        with sqlite3.connect(table_path) as connection:
            return pd.read_sql_query(f"SELECT * FROM {table_name}", connection)

    raise ValueError(f"Unsupported source format: {table_format}")


## 3. Extract all required source tables

We load each operational source into a pandas DataFrame. This makes the later transformation steps easy to follow and easy to debug.


In [22]:
crm_regions = load_source_table("crm_regions")
crm_customers = load_source_table("crm_customers")
erp_products = load_source_table("erp_products")
erp_plants = load_source_table("erp_plants")
hr_employees = load_source_table("hr_employees")
wms_warehouses = load_source_table("wms_warehouses")
so_orders = load_source_table("so_orders")
so_order_items = load_source_table("so_order_items")
mrp_production_orders = load_source_table("mrp_production_orders")
mrp_production_results = load_source_table("mrp_production_results")
wms_inventory_transactions = load_source_table("wms_inventory_transactions")
lgs_shipments = load_source_table("lgs_shipments")
segment_reference = load_source_table("segment_reference")
product_cost_lookup = load_source_table("product_cost_lookup")
production_segment_lookup = load_source_table("production_segment_lookup")
inventory_segment_lookup = load_source_table("inventory_segment_lookup")
sales_employee_segment_lookup = load_source_table("sales_employee_segment_lookup")
customer_region_snapshot = load_source_table("customer_region_snapshot")

for frame_name, frame in {
    "crm_regions": crm_regions,
    "crm_customers": crm_customers,
    "erp_products": erp_products,
    "erp_plants": erp_plants,
    "hr_employees": hr_employees,
    "wms_warehouses": wms_warehouses,
    "so_orders": so_orders,
    "so_order_items": so_order_items,
    "mrp_production_orders": mrp_production_orders,
    "mrp_production_results": mrp_production_results,
    "wms_inventory_transactions": wms_inventory_transactions,
    "lgs_shipments": lgs_shipments,
}.items():
    print(f"{frame_name}: {len(frame)} rows")


crm_regions: 6 rows
crm_customers: 12 rows
erp_products: 10 rows
erp_plants: 4 rows
hr_employees: 8 rows
wms_warehouses: 5 rows
so_orders: 60 rows
so_order_items: 151 rows
mrp_production_orders: 40 rows
mrp_production_results: 40 rows
wms_inventory_transactions: 2000 rows
lgs_shipments: 45 rows


## 4. Standardize data types

Most source files store dates and numbers as text. We convert them into useful pandas types before applying warehouse business rules.


In [23]:
date_columns = {
    "so_orders": ["order_date"],
    "mrp_production_orders": ["plan_start_date", "plan_end_date"],
    "mrp_production_results": ["actual_date"],
    "wms_inventory_transactions": ["txn_date"],
    "lgs_shipments": ["ship_date", "eta_date", "actual_arrival"],
}

dataframes = {
    "so_orders": so_orders,
    "mrp_production_orders": mrp_production_orders,
    "mrp_production_results": mrp_production_results,
    "wms_inventory_transactions": wms_inventory_transactions,
    "lgs_shipments": lgs_shipments,
}

for frame_name, columns in date_columns.items():
    frame = dataframes[frame_name]
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])

numeric_frames = [
    (so_order_items, ["qty_ordered", "unit_price", "discount_pct"]),
    (product_cost_lookup, ["standard_cost"]),
    (mrp_production_orders, ["planned_qty"]),
    (mrp_production_results, ["actual_qty", "scrap_qty", "production_cost"]),
    (wms_inventory_transactions, ["qty_in", "qty_out", "unit_cost"]),
    (lgs_shipments, ["freight_cost"]),
]

for frame, columns in numeric_frames:
    for column in columns:
        frame[column] = pd.to_numeric(frame[column])


## 5. Transform dimensions

Dimensions must be loaded before facts. We follow the same order described in `CONTEXT.md`.


In [24]:
all_dates = pd.concat(
    [
        so_orders[["order_date"]].rename(columns={"order_date": "full_date"}),
        mrp_production_orders[["plan_start_date"]].rename(columns={"plan_start_date": "full_date"}),
        mrp_production_orders[["plan_end_date"]].rename(columns={"plan_end_date": "full_date"}),
        mrp_production_results[["actual_date"]].rename(columns={"actual_date": "full_date"}),
        wms_inventory_transactions[["txn_date"]].rename(columns={"txn_date": "full_date"}),
        lgs_shipments[["ship_date"]].rename(columns={"ship_date": "full_date"}),
        lgs_shipments[["eta_date"]].rename(columns={"eta_date": "full_date"}),
        lgs_shipments[["actual_arrival"]].rename(columns={"actual_arrival": "full_date"}),
    ],
    ignore_index=True,
).dropna()

calendar = pd.date_range(all_dates["full_date"].min(), all_dates["full_date"].max(), freq="D")
dim_time = pd.DataFrame({"full_date": calendar})
dim_time["time_id"] = dim_time["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_time["day_of_week"] = dim_time["full_date"].dt.day_name()
dim_time["month_name"] = dim_time["full_date"].dt.month_name()
dim_time["quarter"] = dim_time["full_date"].dt.quarter.astype(int)
dim_time["year"] = dim_time["full_date"].dt.year.astype(int)
dim_time["full_date"] = dim_time["full_date"].dt.date

dim_region = crm_regions.copy()
dim_region["is_domestic"] = dim_region["country"].eq(manifest["home_country"])

dim_segment = segment_reference.copy()

dim_product = erp_products.rename(
    columns={
        "category": "product_category",
        "uom": "unit_of_measure",
        "principal": "principal_name",
    }
)[[
    "product_id",
    "product_code",
    "product_name",
    "product_category",
    "unit_of_measure",
    "principal_name",
    "target_industry",
]]

dim_customer = crm_customers.rename(columns={"industry": "industry_segment"})[[
    "customer_id",
    "customer_code",
    "customer_name",
    "customer_type",
    "industry_segment",
    "region_id",
]]

segment_lookup = dim_segment[["segment_id", "segment_name"]]
dim_employee = (
    hr_employees
    .merge(segment_lookup, left_on="segment", right_on="segment_name", how="left")
    [["employee_id", "employee_code", "full_name", "department", "segment_id", "region_id"]]
)

dim_warehouse = wms_warehouses.copy()[[
    "warehouse_id",
    "warehouse_code",
    "warehouse_name",
    "warehouse_type",
    "region_id",
]]

dim_plant = erp_plants.copy()[[
    "plant_id",
    "plant_code",
    "plant_name",
    "plant_type",
    "region_id",
]]

display(dim_time.head())
display(dim_product.head())
display(dim_customer.head())


,full_date,time_id,day_of_week,month_name,quarter,year
0,2025-01-01,20250101,Wednesday,January,1,2025
1,2025-01-02,20250102,Thursday,January,1,2025
2,2025-01-03,20250103,Friday,January,1,2025
3,2025-01-04,20250104,Saturday,January,1,2025
4,2025-01-05,20250105,Sunday,January,1,2025


,product_id,product_code,product_name,product_category,unit_of_measure,principal_name,target_industry
0,1,PRD-001,Vitamin C Syrup,Pharma,BOTTLE,BioNusa,Healthcare
1,2,PRD-002,Pain Relief Tablet,Pharma,BOX,BioNusa,Healthcare
2,3,PRD-003,Mineral Water 600ml,Beverage,CASE,FreshWave,Food & Beverage
3,4,PRD-004,Sparkling Drink,Beverage,CASE,FreshWave,Retail
4,5,PRD-005,Household Cleaner,Home Care,BOX,Cleanera,Retail


,customer_id,customer_code,customer_name,customer_type,industry_segment,region_id
0,1,CUST-001,Nusantara Mart,Distributor,Retail,1
1,2,CUST-002,Sehat Farma,Enterprise,Healthcare,1
2,3,CUST-003,Java Wholesale,Distributor,Consumer Goods,2
3,4,CUST-004,Bandung Medika,Enterprise,Healthcare,3
4,5,CUST-005,Lion City Trade,Export,Retail,4


## 6. Transform fact tables

Each fact is created at the grain described in `CONTEXT.md`.


In [25]:
sales_segment = (
    sales_employee_segment_lookup
    .merge(
        dim_segment[["segment_id", "segment_code", "segment_name"]],
        on=["segment_code", "segment_name"],
        how="left",
    )
    [["employee_id", "segment_id"]]
    .rename(columns={"employee_id": "salesperson_id"})
)

customer_region_for_sales = customer_region_snapshot.rename(columns={"region_id": "customer_region_id"})

sales_enriched = (
    so_order_items
    .merge(so_orders, on="order_id", how="left")
    .merge(customer_region_for_sales[["customer_id", "customer_region_id"]], on="customer_id", how="left")
    .merge(sales_segment, on="salesperson_id", how="left")
    .merge(product_cost_lookup[["product_id", "standard_cost"]], on="product_id", how="left")
)

sales_enriched["time_id"] = sales_enriched["order_date"].dt.strftime("%Y%m%d").astype(int)
sales_enriched["net_amount"] = (
    sales_enriched["qty_ordered"]
    * sales_enriched["unit_price"]
    * (1 - sales_enriched["discount_pct"])
).round(2)
sales_enriched["gross_margin_pct"] = (
    (sales_enriched["unit_price"] - sales_enriched["standard_cost"])
    / sales_enriched["unit_price"]
).fillna(0).round(4)

fact_sales = sales_enriched.rename(columns={
    "item_id": "sales_fact_id",
    "salesperson_id": "employee_id",
})[[
    "sales_fact_id",
    "time_id",
    "product_id",
    "customer_id",
    "customer_region_id",
    "segment_id",
    "employee_id",
    "warehouse_id",
    "qty_ordered",
    "unit_price",
    "net_amount",
    "gross_margin_pct",
]].rename(columns={
    "customer_region_id": "region_id",
    "qty_ordered": "quantity_ordered",
})

production_enriched = (
    mrp_production_orders
    .merge(mrp_production_results, on="prod_order_id", how="inner")
    .merge(production_segment_lookup[["prod_order_id", "segment_name"]], on="prod_order_id", how="left")
    .merge(dim_segment[["segment_id", "segment_name"]], on="segment_name", how="left")
)

production_enriched["time_id"] = production_enriched["actual_date"].dt.strftime("%Y%m%d").astype(int)
production_enriched["yield_pct"] = (
    production_enriched["actual_qty"]
    / (production_enriched["actual_qty"] + production_enriched["scrap_qty"])
).round(4)

fact_production = production_enriched.rename(columns={
    "result_id": "production_fact_id",
    "production_cost": "total_production_cost",
})[[
    "production_fact_id",
    "time_id",
    "product_id",
    "plant_id",
    "segment_id",
    "planned_qty",
    "actual_qty",
    "yield_pct",
    "total_production_cost",
]]

inventory_enriched = (
    wms_inventory_transactions
    .merge(inventory_segment_lookup[["txn_id", "segment_name"]], on="txn_id", how="left")
    .merge(dim_segment[["segment_id", "segment_name"]], on="segment_name", how="left")
)

inventory_daily = (
    inventory_enriched
    .assign(txn_day=inventory_enriched["txn_date"].dt.normalize())
    .groupby(["txn_day", "product_id", "warehouse_id", "segment_id"], as_index=False)
    .agg(
        qty_in=("qty_in", "sum"),
        qty_out=("qty_out", "sum"),
        average_unit_cost=("unit_cost", "mean"),
    )
    .sort_values(["product_id", "warehouse_id", "txn_day"])
)

inventory_daily["net_qty_change"] = inventory_daily["qty_in"] - inventory_daily["qty_out"]
inventory_daily["opening_qty"] = (
    inventory_daily.groupby(["product_id", "warehouse_id"])["net_qty_change"]
    .cumsum()
    - inventory_daily["net_qty_change"]
)
inventory_daily["closing_qty"] = inventory_daily["opening_qty"] + inventory_daily["net_qty_change"]
inventory_daily["inventory_value"] = (
    inventory_daily["closing_qty"] * inventory_daily["average_unit_cost"]
).round(2)
inventory_daily["time_id"] = inventory_daily["txn_day"].dt.strftime("%Y%m%d").astype(int)
inventory_daily["inventory_fact_id"] = range(1, len(inventory_daily) + 1)

fact_inventory = inventory_daily[[
    "inventory_fact_id",
    "time_id",
    "product_id",
    "warehouse_id",
    "segment_id",
    "opening_qty",
    "closing_qty",
    "inventory_value",
]]

shipment_enriched = (
    lgs_shipments
    .merge(so_orders[["order_id", "customer_id"]], on="order_id", how="left")
)

shipment_enriched["time_id"] = shipment_enriched["ship_date"].dt.strftime("%Y%m%d").astype(int)
shipment_enriched["on_time_flag"] = shipment_enriched["actual_arrival"] <= shipment_enriched["eta_date"]

fact_shipment = shipment_enriched.rename(columns={
    "shipment_id": "shipment_fact_id",
})[[
    "shipment_fact_id",
    "time_id",
    "customer_id",
    "region_id",
    "warehouse_id",
    "shipping_method",
    "freight_cost",
    "on_time_flag",
]]

display(fact_sales.head())
display(fact_production.head())
display(fact_inventory.head())
display(fact_shipment.head())


,sales_fact_id,time_id,product_id,customer_id,region_id,segment_id,employee_id,warehouse_id,quantity_ordered,unit_price,net_amount,gross_margin_pct
0,1,20250409,1,9,4,4,4,4,120.0,29.14,3321.96,0.3651
1,2,20250409,7,9,4,4,4,4,103.0,100.80,10382.40,0.2560
2,3,20250409,5,9,4,4,4,4,44.0,22.72,949.70,0.3266
3,4,20250409,3,9,4,4,4,4,76.0,29.17,2216.92,0.2424
4,5,20250130,5,12,2,4,4,1,101.0,19.90,1808.91,0.2312


,production_fact_id,time_id,product_id,plant_id,segment_id,planned_qty,actual_qty,yield_pct,total_production_cost
0,1,20250211,4,3,2,769.0,696.59,0.9698,20434.35
1,2,20250322,8,3,4,859.0,761.50,0.9358,54987.00
2,3,20250104,2,2,3,477.0,497.03,0.9634,5780.43
3,4,20250415,5,3,2,1065.0,1100.17,0.9444,16627.78
4,5,20250326,4,1,2,900.0,834.08,0.9612,24519.17


,inventory_fact_id,time_id,product_id,warehouse_id,segment_id,opening_qty,closing_qty,inventory_value
0,1,20250101,1,1,3,0.0,30.0,577.20
50,2,20250103,1,1,3,30.0,-47.0,-911.33
100,3,20250107,1,1,3,-47.0,13.0,245.83
150,4,20250109,1,1,3,13.0,-59.0,-1149.32
200,5,20250113,1,1,3,-59.0,121.0,2286.90


,shipment_fact_id,time_id,customer_id,region_id,warehouse_id,shipping_method,freight_cost,on_time_flag
0,1,20250411,9,4,4,Sea Freight,434.41,True
1,2,20250204,12,2,1,Truck,1550.33,True
2,3,20250223,11,3,1,Air Freight,1519.58,False
3,4,20250212,2,1,4,Air Freight,788.53,False
4,5,20250413,10,1,4,Air Freight,601.82,False


## 7. Prepare load order

The warehouse load order is important because fact tables depend on dimensions.


In [26]:
warehouse_tables = {
    "dim_time": dim_time,
    "dim_region": dim_region,
    "dim_segment": dim_segment,
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_employee": dim_employee,
    "dim_warehouse": dim_warehouse,
    "dim_plant": dim_plant,
    "fact_sales": fact_sales,
    "fact_production": fact_production,
    "fact_inventory": fact_inventory,
    "fact_shipment": fact_shipment,
}

load_sequence = [
    "dim_time",
    "dim_region",
    "dim_segment",
    "dim_product",
    "dim_customer",
    "dim_employee",
    "dim_warehouse",
    "dim_plant",
    "fact_sales",
    "fact_production",
    "fact_inventory",
    "fact_shipment",
]

pd.DataFrame(
    [{"table_name": name, "rows": len(warehouse_tables[name])} for name in load_sequence]
)


,table_name,rows
0,dim_time,122
1,dim_region,6
2,dim_segment,4
3,dim_product,10
4,dim_customer,12
5,dim_employee,8
6,dim_warehouse,5
7,dim_plant,4
8,fact_sales,151
9,fact_production,40


## 8. Load data into the warehouse database

This notebook uses a full refresh approach for learning purposes: it deletes warehouse rows in reverse dependency order, then inserts fresh results in the correct load order.


In [27]:
engine = create_engine(DATABASE_URL)
Base.metadata.create_all(engine)

delete_sequence = list(reversed(load_sequence))

with engine.begin() as connection:
    for table_name in delete_sequence:
        connection.execute(text(f"DELETE FROM {table_name}"))

for table_name in load_sequence:
    warehouse_tables[table_name].to_sql(
        table_name,
        con=engine,
        if_exists="append",
        index=False,
        method="multi",
    )
    print(f"Loaded {table_name}: {len(warehouse_tables[table_name])} rows")


Loaded dim_time: 122 rows
Loaded dim_region: 6 rows
Loaded dim_segment: 4 rows
Loaded dim_product: 10 rows
Loaded dim_customer: 12 rows
Loaded dim_employee: 8 rows
Loaded dim_warehouse: 5 rows
Loaded dim_plant: 4 rows
Loaded fact_sales: 151 rows
Loaded fact_production: 40 rows
Loaded fact_inventory: 2000 rows
Loaded fact_shipment: 45 rows


## 9. Validate the load

The final check reads row counts back from the warehouse database.


In [28]:
validation_rows = []

with engine.connect() as connection:
    for table_name in load_sequence:
        row_count = connection.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar_one()
        validation_rows.append({"table_name": table_name, "row_count": row_count})

pd.DataFrame(validation_rows)


,table_name,row_count
0,dim_time,122
1,dim_region,6
2,dim_segment,4
3,dim_product,10
4,dim_customer,12
5,dim_employee,8
6,dim_warehouse,5
7,dim_plant,4
8,fact_sales,151
9,fact_production,40
